In [14]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, matthews_corrcoef, precision_score, recall_score, roc_auc_score

PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
MODEL_PATH = PROJECT_DIR / 'model' / 'LGBM_model.pkl'

model = joblib.load(MODEL_PATH)
selected_features = model.feature_name_

X_train = pd.read_csv(DATA_DIR / 'X_train.csv').drop(columns='sequence')
X_test = pd.read_csv(DATA_DIR / 'X_test.csv').drop(columns='sequence')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv', index_col=0)['target']
y_test = pd.read_csv(DATA_DIR / 'y_test.csv', index_col=0)['target']

sequence_craft = pd.read_csv(DATA_DIR / 'sequence_craft_clean_emb.csv')
X_sequence_craft = sequence_craft[selected_features]
y_sequence_craft = sequence_craft['target_binary']

X_all = pd.concat([X_train, X_test], ignore_index=True)
y_all = pd.concat([y_train, y_test], ignore_index=True)

In [15]:
def calculate_metrics(X, y, threshold=0.5):
    positive_class_index = list(model.classes_).index(1)
    y_proba = model.predict_proba(X[selected_features])[:, positive_class_index]
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()

    return pd.Series({
        'threshold': threshold,
        'roc/auc': roc_auc_score(y, y_proba),
        'f1': f1_score(y, y_pred, zero_division=0),
        'f1 (macro)': f1_score(y, y_pred, average='macro', zero_division=0),
        'f1 (micro)': f1_score(y, y_pred, average='micro', zero_division=0),
        'precision': precision_score(y, y_pred, zero_division=0),
        'recall': recall_score(y, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y, y_pred),
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp,
    })

In [16]:
pd.DataFrame(
    [
        calculate_metrics(X_all, y_all),
        calculate_metrics(X_test, y_test),
        calculate_metrics(X_test, y_test, threshold=0.90),
        calculate_metrics(X_sequence_craft, y_sequence_craft),
        calculate_metrics(X_sequence_craft, y_sequence_craft, threshold=0.90),
    ],
    index=['Train+Test', 'Test', 'Test (0.95)', 'Sequence_craft', 'Sequence_craft (0.95)'],
)

,threshold,roc/auc,f1,f1 (macro),f1 (micro),precision,recall,mcc,TN,FP,FN,TP
Train+Test,0.5,0.970366,0.900662,0.942905,0.974160,0.964539,0.844720,0.888460,995.0,5.0,25.0,136.0
Test,0.5,0.755752,0.318182,0.623546,0.871245,0.583333,0.218750,0.301930,196.0,5.0,25.0,7.0
Test (0.95),0.9,0.755752,0.117647,0.524101,0.871245,1.000000,0.062500,0.233202,201.0,0.0,30.0,2.0
Sequence_craft,0.5,0.934948,0.880259,0.892572,0.893983,0.918919,0.844720,0.787686,176.0,12.0,25.0,136.0
Sequence_craft (0.95),0.9,0.934948,0.888136,0.903125,0.905444,0.977612,0.813665,0.817643,185.0,3.0,30.0,131.0
